# PATSTAT Citation Counts — C3 / C5 / C10 / C_all, by provenance, and uniqueC

The twin of `PatentView/notebook/patent_citation.ipynb`, read off `patstat_reference.parquet`.

## Metric
For focal application $P$ and window $W$: $C_W(P) = \#\{\text{citation rows } c \to P : 0 \le y_c - y_P \le W\}$
with $y$ = **filing year** (PatentView: grant year); $C_{all}$ = all with $y_c \ge y_P$.
Split by `bucket`: `C_examiner`, `C_applicant`, `C_other` (third-party observations excluded upstream;
`C = C_examiner + C_applicant + C_other` within a window).

**`uniqueC`** = distinct **citing applications**. A citing application cites the same target through
several of its publications (A1 search report, then B1) and against several publications of the target,
so `C` counts one act of citing several times; `uniqueC` counts it once. It is the PATSTAT counterpart of
PatentView's granted + pre-grant de-duplication, and the count the impact analyses use. For a window $W$ a
citing application counts when the **minimum** age over its edges to the target is $\le W$ (windows are
nested, so this is exact); the provenance splits take the minimum over that bucket's edges only.

## Conventions
`ps.EDGE_WHERE` = `age >= 0 AND NOT replenished`.

## Output
`PATSTAT/output/patstat_citation.parquet` — `appln_id` + `C{w}, C_examiner{w}, C_applicant{w}, C_other{w},
uniqueC{w}, uniqueC_examiner{w}, uniqueC_applicant{w}, uniqueC_other{w}` for `w in {_3,_5,_10,_all}`.
Only cited applications appear; an application absent from the file received no citation
(`patstat_hit_probability` fills the zeros).

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_citation.parquet')
ps.preflight('patstat_citation')

REF = ps.out('patstat_reference.parquet')
WINSFX = [(3, '_3'), (5, '_5'), (10, '_10'), (-1, '_all')]
con = ps.connect()
print('edge filter:', ps.EDGE_WHERE)

## 1. Windowed counts (rows) and uniqueC (distinct citing applications)

In [ ]:
%%time
sel_rows, sel_uni = [], []
for w, sfx in WINSFX:
    cond = 'TRUE' if w == -1 else f'age <= {w}'
    sel_rows.append(f'count(*) FILTER (WHERE {cond}) AS C{sfx}')
    for b in ps.BUCKETS:
        sel_rows.append(f"count(*) FILTER (WHERE {cond} AND bucket = '{b}') AS C_{b}{sfx}")
    condu = 'TRUE' if w == -1 else f'd_all <= {w}'
    sel_uni.append(f'count(*) FILTER (WHERE {condu}) AS uniqueC{sfx}')
    for b in ps.BUCKETS:
        cb = f'd_{b} IS NOT NULL' if w == -1 else f'd_{b} <= {w}'
        sel_uni.append(f'count(*) FILTER (WHERE {cb}) AS uniqueC_{b}{sfx}')
con.execute(f"""CREATE OR REPLACE TABLE rows_c AS
  SELECT cited_id AS appln_id, {', '.join(sel_rows)} FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE} GROUP BY 1""")
con.execute(f"""CREATE OR REPLACE TABLE uni_c AS
  WITH e AS (SELECT cited_id, citing_id, min(age) AS d_all,
                    {', '.join(f"min(age) FILTER (WHERE bucket = '{b}') AS d_{b}" for b in ps.BUCKETS)}
             FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE} GROUP BY 1, 2)
  SELECT cited_id AS appln_id, {', '.join(sel_uni)} FROM e GROUP BY 1""")
con.execute(f"""COPY (SELECT * FROM rows_c JOIN uni_c USING (appln_id) ORDER BY appln_id)
  TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
n = pq.ParquetFile(OUT_FP).metadata.num_rows
print(f'WROTE {OUT_FP}  ({n:,} cited applications, {len(pq.ParquetFile(OUT_FP).schema_arrow.names)} cols)')

## 2. Checks and summary

In [ ]:
chk = con.execute(f"""SELECT
  count(*) FILTER (WHERE C_3 > C_5 OR C_5 > C_10 OR C_10 > C_all) AS C_monotonicity_violations,
  count(*) FILTER (WHERE uniqueC_3 > uniqueC_5 OR uniqueC_5 > uniqueC_10 OR uniqueC_10 > uniqueC_all) AS uniqueC_monotonicity_violations,
  count(*) FILTER (WHERE uniqueC_all > C_all) AS uniqueC_gt_C,
  count(*) FILTER (WHERE C_all <> C_examiner_all + C_applicant_all + C_other_all) AS bucket_sum_violations
  FROM read_parquet('{OUT_FP}')""").fetchdf()
display(chk); assert chk.iloc[0].sum() == 0
print(f'{"window":<8}{"mean C":>10}{"mean uniqueC":>14}{"rows per citing app":>21}')
for w, s in WINSFX:
    c, u = con.execute(f"SELECT avg(C{s}), avg(uniqueC{s}) FROM read_parquet('{OUT_FP}')").fetchone()
    print(f'{s.lstrip("_"):<8}{c:>10.3f}{u:>14.3f}{c / u:>21.3f}')
display(con.execute(f"""SELECT 'examiner' AS bucket, sum(C_examiner_all) AS rows, sum(uniqueC_examiner_all) AS citing_apps FROM read_parquet('{OUT_FP}')
  UNION ALL SELECT 'applicant', sum(C_applicant_all), sum(uniqueC_applicant_all) FROM read_parquet('{OUT_FP}')
  UNION ALL SELECT 'other', sum(C_other_all), sum(uniqueC_other_all) FROM read_parquet('{OUT_FP}')""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') ORDER BY uniqueC_all DESC LIMIT 6").fetchdf())
con.close()